In [ ]:
# ================================================================
#                 WORKING PIPELINE
# ================================================================

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def safe_read(path):
    try:
        return pd.read_csv(path)
    except:
        return pd.read_csv(path, encoding="latin1", engine="python")

# Load datasets
df_omar      = safe_read("omar_sample.csv")
df_prince    = safe_read("prince_sample_2007.csv")
df_avantika  = safe_read("avantika_sample.csv")
df_niasia    = safe_read("niasiaflightinfo.csv")
df_attaluri  = safe_read("attaluri_sample.csv")

df = pd.concat([df_omar, df_prince, df_avantika, df_niasia, df_attaluri], ignore_index=True)

print("Combined shape:", df.shape)




Combined shape: (50376, 36)


In [ ]:
# Clean column names
df.columns = df.columns.str.strip().str.replace(" ", "", regex=False)

# Drop fully empty columns
df = df.dropna(axis=1, how="all")

# Fix target
df["Delayed"] = df["Delayed"].astype(str).str.upper()
df = df[df["Delayed"].isin(["Y", "N"])]
df["Delayed"] = df["Delayed"].map({"Y": 1, "N": 0})

# Identify numeric & categorical columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

if "Delayed" in num_cols:
    num_cols.remove("Delayed")

# --------------------------------------------------------------
#  Drop empty numeric columns
# --------------------------------------------------------------
empty_num_cols = [col for col in num_cols if df[col].notna().sum() == 0]
print("Dropping empty numeric columns:", empty_num_cols)

df = df.drop(columns=empty_num_cols)
num_cols = [c for c in num_cols if c not in empty_num_cols]

# --------------------------------------------------------------
#  Drop empty categorical columns
# --------------------------------------------------------------
empty_cat_cols = [col for col in cat_cols if df[col].notna().sum() == 0]
print("Dropping empty categorical columns:", empty_cat_cols)

df = df.drop(columns=empty_cat_cols)
cat_cols = [c for c in cat_cols if c not in empty_cat_cols]



Dropping empty numeric columns: ['DayOfMonth', 'DaysOftheWeek', 'CRSDepTIme', 'FlightNumb']
Dropping empty categorical columns: ['CarrieDelay', 'Case']


In [ ]:
# Impute numeric
num_imputer = SimpleImputer(strategy="median")
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Impute categorical
cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Encode categorical
label_encoders = {}
for col in cat_cols:
    df[col] = df[col].astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Features & target
X = df.drop(columns=["Delayed"])
y = df["Delayed"]

# Scale numeric
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Train model
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

print("\n==================== RESULTS ====================")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("=================================================")


==================== RESULTS ====================
Accuracy: 0.9319

Confusion Matrix:
 [[4440  190]
 [ 497 4968]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.96      0.93      4630
           1       0.96      0.91      0.94      5465

    accuracy                           0.93     10095
   macro avg       0.93      0.93      0.93     10095
weighted avg       0.93      0.93      0.93     10095



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# =====================================================
# 1. LOAD YOUR SAMPLE FILE
# =====================================================
sample_df = pd.read_excel("sample data_project.xlsx")

# Clean column names like training
sample_df.columns = (
    sample_df.columns
        .str.strip()
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
        .str.replace("/", "", regex=False)
)

print("Original sample columns:", sample_df.columns.tolist())


# =====================================================
# 2. REMOVE COLUMNS THAT WERE DROPPED IN TRAINING
# =====================================================
cols_deleted_training = [
    'DayOfMonth', 'DaysOftheWeek', 'CRSDepTIme', 'FlightNumb',
    'CarrieDelay', 'Case'
]

sample_df = sample_df.drop(columns=[c for c in cols_deleted_training if c in sample_df.columns],
                           errors='ignore')


# =====================================================
# 3. REMOVE EXTRA COLUMNS NOT USED FOR TRAINING
# =====================================================
train_cols = X.columns.tolist()   # From your trained pipeline

sample_df = sample_df[[col for col in sample_df.columns if col in train_cols]]


# =====================================================
# 4. ADD ANY MISSING COLUMNS (MATCH TRAINING EXACTLY)
# =====================================================
for col in train_cols:
    if col not in sample_df.columns:
        if col in num_cols:
            sample_df[col] = 0
        else:
            sample_df[col] = label_encoders[col].classes_[0]


# MATCH TRAINING COLUMN ORDER
sample_df = sample_df[train_cols]

print("Aligned sample columns:", sample_df.columns.tolist())


# =====================================================
# 5. IMPUTE USING TRAINED IMPUTERS
# =====================================================
sample_df[num_cols] = num_imputer.transform(sample_df[num_cols])
sample_df[cat_cols] = cat_imputer.transform(sample_df[cat_cols])


# =====================================================
# 6. LABEL ENCODE USING TRAINED ENCODERS
# =====================================================
for col in cat_cols:
    sample_df[col] = sample_df[col].astype(str)

    # Replace unseen labels
    sample_df[col] = sample_df[col].apply(
        lambda x: x if x in label_encoders[col].classes_
        else label_encoders[col].classes_[0]
    )

    sample_df[col] = label_encoders[col].transform(sample_df[col])


# =====================================================
# 7. SCALE NUMERIC COLUMNS
# =====================================================
sample_df[num_cols] = scaler.transform(sample_df[num_cols])


# =====================================================
# 8. PREDICT "DELAYED" YES/NO
# =====================================================
pred = model.predict(sample_df)

# Convert 1 → Yes, 0 → No
sample_df["Delayed"] = ["Yes" if p == 1 else "No" for p in pred]

print(sample_df[["Delayed"]])


# =====================================================
# 9. SAVE OUTPUT EXCEL (WORKING DIRECTORY)
# =====================================================
output_filename = "sample_with_predictions.xlsx"
sample_df.to_excel(output_filename, index=False)

print(f"File saved successfully as: {output_filename}")


Original sample columns: ['Student', 'Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime', 'UniqueCarrier', 'FlightNum', 'TailNum', 'ActualElapsedTime', 'CRSElapsedTime', 'AirTime', 'ArrDelay', 'DepDelay', 'Origin', 'Dest', 'Distance', 'TaxiIn', 'TaxiOut', 'Cancelled', 'Delayed']
Aligned sample columns: ['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime', 'UniqueCarrier', 'FlightNum', 'TailNum', 'ActualElapsedTime', 'CRSElapsedTime', 'AirTime', 'ArrDelay', 'DepDelay', 'Origin', 'Dest', 'Distance', 'TaxiIn', 'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
  Delayed
0      No
1      No
2      No
3      No
4      No
5      No
6      No
7      No
8      No
9      No
File saved successfully as: sample_with_predictions.xlsx
